# Fine Tune Qwen Model For Python Code Generation

In [2]:
%pip install -q transformers datasets peft accelerate bitsandbytes trl pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 44.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


### dataset = https://www.kaggle.com/datasets/mirajverisk/code-dataset/data

In [3]:
model_id = "Qwen/Qwen3-0.6B"

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [5]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for message in messages[::-1] %}
    {%- set index = (messages|length - 

In [6]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.special_tokens_map)

{'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}


# Import Dataset

In [7]:
import pandas as pd

df = pd.read_csv(
    r"/kaggle/input/datasets/mirajverisk/code-dataset/python_100k_dataset.tsv",
    sep="\t"
)

# Unescape literal \n, \t, \r stored as 2-char strings in the TSV
for col in ["input", "output"]:
    df[col] = (
        df[col].astype(str)
               .str.replace(r"\\n", "\n", regex=True)
               .str.replace(r"\\t", "\t", regex=True)
               .str.replace(r"\\r", "\r", regex=True)
    )

print("Shape:", df.shape)
df.head(3)

Shape: (100000, 2)


,input,output
0,Create a function to calculate the sum of a se...,def sum_sequence(sequence):\n sum = 0\n for ...
1,Develop a function that will add two strings,"def add_strings(str1, str2):\n """"""This func..."
2,Implement a sorting algorithm to sort a given ...,def bubble_sort(arr):\n n = len(arr)\n \n ...


# Split the DataSet

In [8]:
df = df.head(20000)

In [9]:
from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)
print(len(train_df), len(eval_df))

16000 4000


# Build instruction-style prompts (Qwen chat template)

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [11]:
SYSTEM_PROMPT = (
    "You are an expert Python software engineer. "
    "Generate accurate, efficient, and readable Python code. "
    "Output only executable Python code without explanations, markdown, or additional text."
)

def build_messages(sentence, answer=None):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': sentence},
    ]
    if answer is not None:
        msgs.append({'role': 'assistant', 'content': answer})
    return msgs


In [12]:
row = df.iloc[0]
row

input     Create a function to calculate the sum of a se...
output    def sum_sequence(sequence):\n  sum = 0\n  for ...
Name: 0, dtype: object

In [13]:
messages = build_messages(
    row["input"],
    row["output"]
)

print(messages)

[{'role': 'system', 'content': 'You are an expert Python software engineer. Generate accurate, efficient, and readable Python code. Output only executable Python code without explanations, markdown, or additional text.'}, {'role': 'user', 'content': 'Create a function to calculate the sum of a sequence of integers.'}, {'role': 'assistant', 'content': 'def sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum'}]


In [14]:
# Preview one
sample_txt = tokenizer.apply_chat_template(
    messages,
    tokenize=False)
    
print(sample_txt)

<|im_start|>system
You are an expert Python software engineer. Generate accurate, efficient, and readable Python code. Output only executable Python code without explanations, markdown, or additional text.<|im_end|>
<|im_start|>user
Create a function to calculate the sum of a sequence of integers.<|im_end|>
<|im_start|>assistant
<think>

</think>

def sum_sequence(sequence):
  sum = 0
  for num in sequence:
    sum += num
  return sum<|im_end|>



# Tokenize with prompt masking

In [15]:
print(tokenizer.model_max_length)

131072


In [16]:
import numpy as np

# Sample 5,000 examples (change if you want)
sample_df = df.sample(n=min(5000, len(df)), random_state=42)

texts = [
    tokenizer.apply_chat_template(
        build_messages(inp, out),
        tokenize=False,
        add_generation_prompt=False,
    )
    for inp, out in zip(sample_df["input"], sample_df["output"])
]

encodings = tokenizer(texts, add_special_tokens=False)

lengths = [len(ids) for ids in encodings["input_ids"]]

print(f"Min: {np.min(lengths)}")
print(f"Mean: {np.mean(lengths):.2f}")
print(f"Median: {np.median(lengths)}")
print(f"90th percentile: {np.percentile(lengths, 90)}")
print(f"95th percentile: {np.percentile(lengths, 95)}")
print(f"99th percentile: {np.percentile(lengths, 99)}")
print(f"Max: {np.max(lengths)}")

for max_len in [256, 512, 1024, 2048]:
    truncated = sum(l > max_len for l in lengths)
    print(
        f"MAX_LEN={max_len}: "
        f"{truncated}/{len(lengths)} "
        f"({truncated / len(lengths) * 100:.2f}%) examples would be truncated"
    )

Min: 60
Mean: 198.78
Median: 153.0
90th percentile: 305.10000000000036
95th percentile: 386.0500000000002
99th percentile: 894.0600000000013
Max: 4107
MAX_LEN=256: 780/5000 (15.60%) examples would be truncated
MAX_LEN=512: 121/5000 (2.42%) examples would be truncated
MAX_LEN=1024: 45/5000 (0.90%) examples would be truncated
MAX_LEN=2048: 23/5000 (0.46%) examples would be truncated


#  add_generation_prompt (True and False Difference)

In [17]:
sample_df["input"]

10650    Create a decision tree algorithm to predict th...
2041     Develop an algorithm to generate a unique, ran...
8668     Create a class to store and show a student's r...
1114     Write a Python function to generate a random s...
13902    Write a function in Python which converts a gi...
                               ...                        
3761     Generate a set of 101 Fibonacci sequence numbers.
5478     write a program to calculate the sum of all no...
5805                       Invent a new sorting algorithm.
10084    Generate a Python program to accept a string a...
13494    Generate a Python program to print the element...
Name: input, Length: 5000, dtype: object

In [18]:
sample_txt = tokenizer.apply_chat_template(
    build_messages(sample_df["input"].iloc[0]),
    tokenize=False
)
print(sample_txt)

<|im_start|>system
You are an expert Python software engineer. Generate accurate, efficient, and readable Python code. Output only executable Python code without explanations, markdown, or additional text.<|im_end|>
<|im_start|>user
Create a decision tree algorithm to predict the quality of wine.<|im_end|>



In [19]:
sample_txt = tokenizer.apply_chat_template(
    build_messages(sample_df["input"].iloc[0]),
    tokenize=False,
    add_generation_prompt=True
)

print(sample_txt)

<|im_start|>system
You are an expert Python software engineer. Generate accurate, efficient, and readable Python code. Output only executable Python code without explanations, markdown, or additional text.<|im_end|>
<|im_start|>user
Create a decision tree algorithm to predict the quality of wine.<|im_end|>
<|im_start|>assistant



`add_generation_prompt=True appends the beginning of a new assistant message (<|im_start|>assistant) to the end of the conversation, indicating where the model should start generating its response.`

# Understanding Input Masking

`we are hiding the input tokens from the loss calculation`

In [20]:
print(sample_df["input"].iloc[0])
print("="*215)
print(sample_df["output"].iloc[0])

Create a decision tree algorithm to predict the quality of wine.
# Importing the necessary libraries 
from sklearn import tree 

# Feature extraction 
X = wine_data.drop('quality', axis = 1)

# Initialising the Decision Tree 
classifier = tree.DecisionTreeClassifier()

# Fitting the Decision Tree to the dataset 
classifier.fit(X, y) 

# Predicting the results 
y_pred = classifier.predict(X_test)


In [21]:
prompt_ids = tokenizer.apply_chat_template(
    build_messages(sample_df["input"].iloc[0]),
    tokenize=True,
    add_generation_prompt=True
)

prompt_ids

{'input_ids': [151644, 8948, 198, 2610, 525, 458, 6203, 13027, 3162, 23576, 13, 19813, 13382, 11, 11050, 11, 323, 33798, 13027, 2038, 13, 9258, 1172, 32156, 13027, 2038, 2041, 40841, 11, 50494, 11, 476, 5107, 1467, 13, 151645, 198, 151644, 872, 198, 4021, 264, 5480, 4916, 12111, 311, 7023, 279, 4271, 315, 13078, 13, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [22]:
full_ids = tokenizer.apply_chat_template(
    build_messages(sample_df["input"].iloc[0], sample_df["output"].iloc[0]),
    tokenize=True
)

full_ids

{'input_ids': [151644, 8948, 198, 2610, 525, 458, 6203, 13027, 3162, 23576, 13, 19813, 13382, 11, 11050, 11, 323, 33798, 13027, 2038, 13, 9258, 1172, 32156, 13027, 2038, 2041, 40841, 11, 50494, 11, 476, 5107, 1467, 13, 151645, 198, 151644, 872, 198, 4021, 264, 5480, 4916, 12111, 311, 7023, 279, 4271, 315, 13078, 13, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 2, 13213, 287, 279, 5871, 20186, 715, 1499, 17987, 1159, 4916, 4710, 2, 19998, 32189, 715, 55, 284, 13078, 1769, 19086, 492, 10473, 516, 8024, 284, 220, 16, 692, 2, 4127, 3791, 279, 40425, 8942, 715, 78100, 284, 4916, 22442, 1816, 6533, 33895, 2822, 2, 434, 14810, 279, 40425, 8942, 311, 279, 10337, 715, 78100, 20860, 7644, 11, 379, 8, 4710, 2, 32710, 287, 279, 3059, 715, 88, 12830, 284, 33365, 23772, 7644, 4452, 8, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [23]:
MAX_LEN = 500

print("before:", len(full_ids['input_ids']))

full_ids = full_ids['input_ids'][:MAX_LEN]

print("after:", len(full_ids))

before: 138
after: 138


In [24]:
prompt_ids

{'input_ids': [151644, 8948, 198, 2610, 525, 458, 6203, 13027, 3162, 23576, 13, 19813, 13382, 11, 11050, 11, 323, 33798, 13027, 2038, 13, 9258, 1172, 32156, 13027, 2038, 2041, 40841, 11, 50494, 11, 476, 5107, 1467, 13, 151645, 198, 151644, 872, 198, 4021, 264, 5480, 4916, 12111, 311, 7023, 279, 4271, 315, 13078, 13, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [25]:
prompt_ids = prompt_ids["input_ids"]
prompt_length = len(prompt_ids)
prompt_length

57

In [26]:
labels = [-100] * prompt_length
labels[:10]

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

In [27]:
full_ids[prompt_length:]

[151667,
 271,
 151668,
 271,
 2,
 13213,
 287,
 279,
 5871,
 20186,
 715,
 1499,
 17987,
 1159,
 4916,
 4710,
 2,
 19998,
 32189,
 715,
 55,
 284,
 13078,
 1769,
 19086,
 492,
 10473,
 516,
 8024,
 284,
 220,
 16,
 692,
 2,
 4127,
 3791,
 279,
 40425,
 8942,
 715,
 78100,
 284,
 4916,
 22442,
 1816,
 6533,
 33895,
 2822,
 2,
 434,
 14810,
 279,
 40425,
 8942,
 311,
 279,
 10337,
 715,
 78100,
 20860,
 7644,
 11,
 379,
 8,
 4710,
 2,
 32710,
 287,
 279,
 3059,
 715,
 88,
 12830,
 284,
 33365,
 23772,
 7644,
 4452,
 8,
 151645,
 198]

In [28]:
labels = labels + full_ids[prompt_length:]
labels

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 151667,
 271,
 151668,
 271,
 2,
 13213,
 287,
 279,
 5871,
 20186,
 715,
 1499,
 17987,
 1159,
 4916,
 4710,
 2,
 19998,
 32189,
 715,
 55,
 284,
 13078,
 1769,
 19086,
 492,
 10473,
 516,
 8024,
 284,
 220,
 16,
 692,
 2,
 4127,
 3791,
 279,
 40425,
 8942,
 715,
 78100,
 284,
 4916,
 22442,
 1816,
 6533,
 33895,
 2822,
 2,
 434,
 14810,
 279,
 40425,
 8942,
 311,
 279,
 10337,
 715,
 78100,
 20860,
 7644,
 11,
 379,
 8,
 4710,
 2,
 32710,
 287,
 279,
 3059,
 715,
 88,
 12830,
 284,
 33365,
 23772,
 7644,
 4452,
 8,
 151645,
 198]

In [29]:
from datasets import Dataset

MAX_LEN = 1024

def encode(example):

    prompt_ids = tokenizer.apply_chat_template(
        build_messages(example['input']),
        tokenize=True,
        add_generation_prompt=True
    )

    prompt_ids = prompt_ids["input_ids"]


    full_ids = tokenizer.apply_chat_template(
        build_messages(example['input'], example['output']),
        tokenize=True
    )

    full_ids = full_ids["input_ids"]


    full_ids = full_ids[:MAX_LEN]


    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = [-100] * prompt_len
    labels = labels + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids) #since we dont have padding yet
    }


In [ ]:
train_ds = Dataset.from_pandas(train_df[['input','output']]).map(encode)

eval_ds = Dataset.from_pandas(eval_df[['input','output']]).map(encode)

print(train_ds)

# Understanding the Data Collator

### Before Collator

```python
[
    {
        "input_ids": [1, 2, 3, 4],
        "labels": [-100, -100, 3, 4],
        "attention_mask": [1, 1, 1, 1]
    },
    {
        "input_ids": [1, 2],
        "labels": [-100, 2],
        "attention_mask": [1, 1]
    }
]
```

### After Collator

```python
input_ids:
[
    [1, 2, 3, 4],
    [1, 2, pad, pad]
]

labels:
[
    [-100, -100, 3, 4],
    [-100, 2, -100, -100]
]

attention_mask:
[
    [1, 1, 1, 1],
    [1, 1, 0, 0]
]
```

### Padding Rules

| Field | Padding Value |
|---|---|
| `input_ids` | `pad_token_id` |
| `labels` | `-100` |
| `attention_mask` | `0` |

In [31]:
from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True
)

# Load model + attach LoRA adapters

In [32]:
!pip install -U -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.0 MB/s eta 0:00:00a 0:00:01


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

device = 'cuda' if torch.cuda.is_available() else 'cpu'

dtype = torch.bfloat16 if device == 'cuda' else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=dtype,
    trust_remote_code=True
).to(device)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_cfg)

model.print_trainable_parameters()

# Train

In [34]:
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [35]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='./qwen_code_gen',

    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    warmup_steps=50,

    lr_scheduler_type='cosine',

    optim="paged_adamw_8bit",

    logging_steps=25,

    save_steps=25,
    eval_strategy='steps',
    eval_steps=25,

    save_total_limit=2,

    bf16=True,
    fp16=False,

    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    processing_class=tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
25,0.981637,0.749028
50,0.733180,0.701063
75,0.690495,0.684241
100,0.720987,0.674532
125,0.691924,0.667167
150,0.720774,0.661808
175,0.685383,0.658164
200,0.659823,0.655879
225,0.680326,0.654690
250,0.681999,0.654405


TrainOutput(global_step=250, training_loss=0.7246528778076172, metrics={'train_runtime': 14897.9148, 'train_samples_per_second': 1.074, 'train_steps_per_second': 0.017, 'total_flos': 2.2981809087184896e+16, 'train_loss': 0.7246528778076172, 'epoch': 1.0})

In [41]:
# import pandas as pd

# df = pd.DataFrame(trainer.state.log_history)

# loss_df = df[["step", "loss"]].dropna()

# loss_df

In [42]:
model.save_pretrained("./Quen_0.6B_PythonCode_Instruct_")
tokenizer.save_pretrained("./Quen_0.6B_PythonCode_Instruct_")

('./Quen_0.6B_PythonCode_Instruct_/tokenizer_config.json',
 './Quen_0.6B_PythonCode_Instruct_/chat_template.jinja',
 './Quen_0.6B_PythonCode_Instruct_/tokenizer.json')

# Inference

In [43]:
model.config.use_cache = True
model.eval()

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

def predict(prompt_text, max_new_tokens=200):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=IM_END_ID
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    code = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return code


tests = [
    "Create a function to calculate the sum of a sequence of integers.",
    "Write a Python function to check if a number is prime.",
    "Create a class for a simple bank account."
]

for t in tests:
    print("INPUT:")
    print(t)

    print("\nOUTPUT:")
    print(predict(t))

    print("\n" + "-"*50)

INPUT:
Create a function to calculate the sum of a sequence of integers.

OUTPUT:
<think>

</think>

def sum_sequence(sequence):
    total = 0
    for num in sequence:
        total += num
    return total

--------------------------------------------------
INPUT:
Write a Python function to check if a number is prime.

OUTPUT:
<think>

</think>

def is_prime(num):
    for i in range(2,num):
        if num %i == 0:
            return False
    else:
        return True

--------------------------------------------------
INPUT:
Create a class for a simple bank account.

OUTPUT:
<think>

</think>

class BankAccount:
    def __init__(self):
        self.balance = 0
    
    def deposit(self, amount):
        self.balance += amount
    
    def withdraw(self, amount):
        if amount > self.balance:
            print("Insufficient balance.")
        else:
            self.balance -= amount

--------------------------------------------------


# Upload Model to Hugging Face

In [44]:
from huggingface_hub import login

login("")  # enter your HF token

In [ ]:
model.push_to_hub(
    "mirajbhandari/Code_Quen_Python"
)

tokenizer.push_to_hub(
    "mirajbhandari/Code_Quen_Python"
)

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "mirajbhandari/Code_Quen_Python"

SYSTEM_PROMPT = (
    "You are an expert Python software engineer. "
    "Generate accurate, efficient, and readable Python code. "
    "Output only executable Python code without explanations, markdown, or additional text."
)

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

In [50]:
def predict(prompt_text, max_new_tokens=500):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=IM_END_ID,
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return re.sub(
        r"<think>.*?</think>\s*",
        "",
        response,
        flags=re.DOTALL
    ).strip()

In [63]:
ques = "Write a Python program to find the sum of all odd numbers in a list."

ans  = predict(ques, max_new_tokens=500)


In [64]:
print(ans)

def sum_odd_numbers(list):
    total = 0
    for num in list:
        if num % 2 != 0:
            total += num
    return total

list = [1, 3, 5, 7]
print(sum_odd_numbers(list)) # 16


In [65]:
def sum_odd_numbers(list):
    total = 0
    for num in list:
        if num % 2 != 0:
            total += num
    return total

list = [1, 3, 5, 7]
print(sum_odd_numbers(list)) # 16

16
